In [15]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold,GridSearchCV
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import make_column_selector,make_column_transformer
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

In [6]:
df = pd.read_csv('E:/Machine_Learning/Datasets/Sacremento.csv',index_col=0)
df

,zip,beds,baths,sqft,type,price,latitude,longitude
city,,,,,,,,
SACRAMENTO,z95838,2,1.0,836,Residential,59222,38.631913,-121.434879
SACRAMENTO,z95823,3,1.0,1167,Residential,68212,38.478902,-121.431028
SACRAMENTO,z95815,2,1.0,796,Residential,68880,38.618305,-121.443839
SACRAMENTO,z95815,2,1.0,852,Residential,69307,38.616835,-121.439146
SACRAMENTO,z95824,2,1.0,797,Residential,81900,38.519470,-121.435768
...,...,...,...,...,...,...,...,...
SACRAMENTO,z95829,4,3.0,2280,Residential,232425,38.457679,-121.359620
SACRAMENTO,z95823,3,2.0,1477,Residential,234000,38.499893,-121.458890
CITRUS_HEIGHTS,z95610,3,2.0,1216,Residential,235000,38.708824,-121.256803


In [7]:
df.drop('zip',axis=1,inplace=True)

In [10]:
df['type'].value_counts()

type
Residential     866
Condo            53
Multi_Family     13
Name: count, dtype: int64

In [13]:
ohe = OneHotEncoder(sparse_output=False,drop='first')
ct = make_column_transformer(
    (ohe,make_column_selector(dtype_include=object)),
    remainder='passthrough',
    verbose_feature_names_out=False).set_output(transform='pandas')

In [9]:
X,y = df.drop('price',axis=1),df['price']

In [19]:
X_ohe = ct.fit_transform(X)
X_ohe.columns

Index(['type_Multi_Family', 'type_Residential', 'beds', 'baths', 'sqft',
       'latitude', 'longitude'],
      dtype='object')

In [20]:
X_train,X_test,y_train,y_test = train_test_split(X_ohe,y,random_state=26,test_size=0.3)

In [14]:
en = ElasticNet()
rf = RandomForestRegressor(random_state=26)
xgb = XGBRegressor()
lgm = LGBMRegressor(random_state=26)
kfold = KFold(n_splits=5,random_state=26,shuffle=True)

In [17]:
en.get_params()

{'alpha': 1.0,
 'copy_X': True,
 'fit_intercept': True,
 'l1_ratio': 0.5,
 'max_iter': 1000,
 'positive': False,
 'precompute': False,
 'random_state': None,
 'selection': 'cyclic',
 'tol': 0.0001,
 'warm_start': False}

In [38]:
params_en = {
    'alpha':np.linspace(0.01,10,10),
    'l1_ratio':np.linspace(0,1,10)
}

gcv = GridSearchCV(en,params_en,cv=kfold,n_jobs=-1)
gcv.fit(X_train,y_train)
print("Best param : ", gcv.best_params_)
print("Best score : ", gcv.best_score_)

Best param :  {'alpha': np.float64(0.01), 'l1_ratio': np.float64(0.8888888888888888)}
Best score :  0.630860628793029


In [24]:
rf.get_params()

{'bootstrap': True,
 'ccp_alpha': 0.0,
 'criterion': 'squared_error',
 'max_depth': None,
 'max_features': 1.0,
 'max_leaf_nodes': None,
 'max_samples': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'monotonic_cst': None,
 'n_estimators': 100,
 'n_jobs': None,
 'oob_score': False,
 'random_state': 26,
 'verbose': 0,
 'warm_start': False}

In [39]:
params_rf = {
    'max_depth':[None,2,3,4,5,6,7],
    'min_samples_leaf':[1,4,6,8,10],
    'min_samples_split':[2,4,6,8,10],
    'n_estimators':[25,50,100]
}
gcv = GridSearchCV(rf,params_rf,cv=kfold,n_jobs=-1)
gcv.fit(X_train,y_train)
print("Best param : ", gcv.best_params_)
print("Best score : ", gcv.best_score_)

Best param :  {'max_depth': 7, 'min_samples_leaf': 4, 'min_samples_split': 2, 'n_estimators': 100}
Best score :  0.6428309710540215


In [30]:
params_xg = {
    'learning_rate' : np.linspace(0.01,0.9,10),
    'max_depth' : [None,2,3,4],
    'n_estimators' : [25,50,100]
}
gcv = GridSearchCV(xgb,params_xg, cv=kfold,n_jobs=-1)
gcv.fit(X_train,y_train)
print("Best param : ", gcv.best_params_)
print("Best score : ", gcv.best_score_)

Best param :  {'learning_rate': np.float64(0.10888888888888888), 'max_depth': 2, 'n_estimators': 100}
Best score :  0.6453517079353333


In [32]:
lgm.get_params()

{'boosting_type': 'gbdt',
 'class_weight': None,
 'colsample_bytree': 1.0,
 'importance_type': 'split',
 'learning_rate': 0.1,
 'max_depth': -1,
 'min_child_samples': 20,
 'min_child_weight': 0.001,
 'min_split_gain': 0.0,
 'n_estimators': 100,
 'n_jobs': None,
 'num_leaves': 31,
 'objective': None,
 'random_state': 26,
 'reg_alpha': 0.0,
 'reg_lambda': 0.0,
 'subsample': 1.0,
 'subsample_for_bin': 200000,
 'subsample_freq': 0}

In [37]:
params_lgm = {
    'max_depth': [-1,2,3,4,5],
    'learning_rate': np.linspace(0.01,0.9,10)
}
gcv = GridSearchCV(lgm,params_lgm, cv=kfold,n_jobs=-1)
gcv.fit(X_train,y_train)
print("Best param : ", gcv.best_params_)
print("Best score : ", gcv.best_score_)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000261 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 672
[LightGBM] [Info] Number of data points in the train set: 652, number of used features: 6
[LightGBM] [Info] Start training from score 239835.892638
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain